# MR-IMSRG(2) Flow Equation Derivation

This notebook uses `qcombo.easyCombo` to derive the **MR-IMSRG(2) flow equations**.

## Theoretical Background

In the IMSRG (In-Medium Similarity Renormalization Group) framework, the Hamiltonian $H(s)$ evolves with the flow parameter $s$ according to:

$$\frac{dH(s)}{ds} = [\eta(s), H(s)]$$

where $\eta(s)$ is the **generator** (here we use the Brillouin generator $\eta = [H, A]$).

At the **IMSRG(2)** truncation level, we keep operators up to 2-body:

$$H(s) = E(s) + \sum_{ij} f^i_j(s) \{a^\dagger_i a_j\} + \frac{1}{4}\sum_{ijkl} \Gamma^{ij}_{kl}(s) \{a^\dagger_i a^\dagger_j a_l a_k\}$$

The flow equations for the 0-body ($E$), 1-body ($f$), and 2-body ($\Gamma$) matrix elements are obtained by evaluating:

- **0-body flow** ($dE/ds$): $[\eta, H]_{0B}$
- **1-body flow** ($df/ds$): $[\eta, H]_{1B}$
- **2-body flow** ($d\Gamma/ds$): $[\eta, H]_{2B}$


**Reference**: H. Hergert, *In-Medium Similarity Renormalization Group for Closed and Open-Shell Nuclei*, Eq. (49)-(51)

In [1]:
# Import qcombo and necessary utility functions
import qcombo
from IPython.display import display, Latex
from sympy import IndexedBase
from sympy import preorder_traversal
from sympy.tensor.indexed import Indexed
import time
from qcombo.simplify import filterLambdaBody

# Define tensor symbols
A = IndexedBase('A')       # generator
G = IndexedBase('G')       # left operator (in easyCombo)
H = IndexedBase('H')       # right operator (in easyCombo)
f = IndexedBase('f')       # one-body matrix element
Gamma = IndexedBase(915)   # Γ two-body matrix element
lamda = IndexedBase(955)   # λ (lambda) density matrix
n = IndexedBase('n')       # occupation number
eta = IndexedBase(951)     # η generator matrix element
eta2 = IndexedBase(952)    # η two-body generator matrix element

print(f"qcombo version: {qcombo.__version__}")

qcombo version: 0.2.0


In [2]:
# Helper function: display SymPy expressions as LaTeX in Jupyter
def jupyterDisplay(expr, title=None):
    """
    Display SymPy expression in LaTeX format in Jupyter Notebook
    """
    if expr == 0 or expr is None:
        display(Latex(f"$$0$$"))
        return
    latex_expr = qcombo.texExp(expr)
    if title:
        print(title)
    display(Latex(f"$${latex_expr}$$"))

# Utility: find tensor in expression
def find_H_tensor(expr):
    """Find the H tensor in an expression"""
    for term in preorder_traversal(expr):
        if isinstance(term, Indexed) and term.base == IndexedBase('H'):
            return term
    return None

def find_A_tensor(expr):
    """Find the A tensor in an expression"""
    for term in preorder_traversal(expr):
        if isinstance(term, Indexed) and term.base == IndexedBase('A'):
            return term
    return None


def find_G_tensor(expr):
    """Find the G tensor in an expression"""
    for term in preorder_traversal(expr):
        if isinstance(term, Indexed) and term.base == IndexedBase('G'):
            return term
    return None

def replace_G_Base(expr, newBase):
    """Replace the base of all G tensors with a new base"""
    G_tensor = find_G_tensor(expr)
    if G_tensor is None:
        return expr
    else:
        return expr.xreplace({G_tensor.base: IndexedBase(newBase)})

def replace_H_Base(expr, newBase):
    """Replace the base of all G tensors with a new base"""
    G_tensor = find_H_tensor(expr)
    if G_tensor is None:
        return expr
    else:
        return expr.xreplace({G_tensor.base: IndexedBase(newBase)})


def re_antisymmetry(expr):
    """
    Re-index the expression.
    e.g. A[i,j]*B[k,l] -> A[a,b]*B[c,d]
    """
    canon_expr = qcombo.canonical.canonicalize(expr.expand(),parallel=False,show_process=False)
    reIndices_expr,indices_set =qcombo.tools.indicesMultToSimp(canon_expr,parallel=False,show_process=False)

    # For multi-body expressions, we also need to restore index antisymmetry
    re_AntiSymetry_expr = qcombo.tools.antisymmetrize_expr(reIndices_expr)

    simplified_expr = qcombo.simplifyUseBoth(re_AntiSymetry_expr.expand(),show_process=False,parallel=False)
    united_expr = qcombo.MergeSameMatrixElement(simplified_expr)
    return united_expr


print("All helper functions defined.")

All helper functions defined.


---
## 1. Zero-body Flow: $dE/ds$

The 0-body flow equation comes from $[\eta, H]_{0B}$.

Since $\eta = \eta_{1B} + \eta_{2B}$ and $H = E + f + \Gamma$, the 0-body contributions are:


In [3]:
# dE comes from [1,1]_0, [1,2]_0, [2,1]_0, [2,2]_0

# Step 1: Compute [η_2B, Γ] -> 0B flow
# This is [2B, 2B] commutator contracted to 0-body
print("="*60)
print("Computing zero-body flow...")
print("="*60)

t0 = time.time()
comm_110 = qcombo.easyCombo(1, 1, 0, parallel=False, show_process=False, savefile=False)
comm_120 = qcombo.easyCombo(1, 2, 0, parallel=False, show_process=False, savefile=False)
comm_210 = qcombo.easyCombo(2, 1, 0, parallel=False, show_process=False, savefile=False)
comm_220 = qcombo.easyCombo(2, 2, 0, parallel=False, show_process=False, savefile=False)
t1 = time.time()

print(f"Computation time: {t1-t0:.2f}s")

Computing zero-body flow...
Computation time: 1.80s


In [4]:
comm_110_expr = 0
comm_120_expr = 0
comm_210_expr = 0
comm_220_expr = 0

for key, value in comm_110.expr_dict.items():
    comm_110_expr += value

for key, value in comm_120.expr_dict.items():
    comm_120_expr += value

for key, value in comm_210.expr_dict.items():
    comm_210_expr += value

for key, value in comm_220.expr_dict.items():
    comm_220_expr += value

# print('commutator [1,1]-0 expression:')
# jupyterDisplay(comm_110_expr)
# print('commutator [1,2]-0 expression:')
# jupyterDisplay(comm_120_expr)
# print('commutator [2,1]-0 expression:')
# jupyterDisplay(comm_210_expr)
# print('commutator [2,2]-0 expression:')
# jupyterDisplay(comm_220_expr)


In [5]:
# Restore index antisymmetry of the expressions
comm_110_expr = re_antisymmetry(comm_110_expr)
comm_120_expr = re_antisymmetry(comm_120_expr)
comm_210_expr = re_antisymmetry(comm_210_expr)
comm_220_expr = re_antisymmetry(comm_220_expr)

# print('commutator [1,1]-0 expression:')
# jupyterDisplay(comm_110_expr)
# print('commutator [1,2]-0 expression:')
# jupyterDisplay(comm_120_expr)
# print('commutator [2,1]-0 expression:')
# jupyterDisplay(comm_210_expr)
# print('commutator [2,2]-0 expression:')
# jupyterDisplay(comm_220_expr)

In [6]:
# Replace G -> η (generator), H -> Γ to obtain the flow equation
#add proper coeffient
dE_ds_110_expr = replace_H_Base(replace_G_Base(comm_110_expr, r'\eta'), "f")
dE_ds_120_expr = replace_H_Base(replace_G_Base(comm_120_expr, r'\eta'), r'\Gamma')/4
dE_ds_210_expr = replace_H_Base(replace_G_Base(comm_210_expr, r'\eta'), "f")/4
dE_ds_220_expr = replace_H_Base(replace_G_Base(comm_220_expr, r'\eta'), r'\Gamma')/16

# jupyterDisplay(dE_ds_110_expr)
# jupyterDisplay(dE_ds_120_expr)
# jupyterDisplay(dE_ds_210_expr)
# jupyterDisplay(dE_ds_220_expr)

In [7]:
print("dE/ds flow equation contains:")
display(Latex(r'$$dE =  $$'))
# 110
print('commutator [1,1]-0, lambda_1B:')
jupyterDisplay(dE_ds_110_expr)
# 120
print('commutator [1,2]-0, lambda_2B:')
jupyterDisplay(dE_ds_120_expr)
# 210
print('commutator [2,1]-0, lambda_2B:')
jupyterDisplay(dE_ds_210_expr)
# 220
print('commutator [2,2]-0, lambda_1B:')
jupyterDisplay(filterLambdaBody(dE_ds_220_expr,1,False,False))
print('commutator [2,2]-0, lambda_2B:')
jupyterDisplay(filterLambdaBody(dE_ds_220_expr,2,False,False))
print('commutator [2,2]-0, lambda_3B:')
jupyterDisplay(filterLambdaBody(dE_ds_220_expr,3,False,False))


dE/ds flow equation contains:


<IPython.core.display.Latex object>

commutator [1,1]-0, lambda_1B:


<IPython.core.display.Latex object>

commutator [1,2]-0, lambda_2B:


...

<IPython.core.display.Latex object>

commutator [2,2]-0, lambda_2B:


<IPython.core.display.Latex object>

commutator [2,2]-0, lambda_3B:


<IPython.core.display.Latex object>

## $dE/ds$ in Reference

$$
\frac{dE}{ds} = \sum_{ab} (n_a - n_b) \eta_b^a f_a^b + \frac{1}{4} \sum_{abcd} \left( \eta_{cd}^{ab} \Gamma_{ab}^{cd} - \Gamma_{cd}^{ab} \eta_{ab}^{cd} \right) n_a n_b \bar{n}_c \bar{n}_d $$
$$+ \frac{1}{4} \sum_{abcd} \left( \frac{d}{ds} \Gamma_{cd}^{ab} \right) \lambda_{cd}^{ab} + \frac{1}{4} \sum_{abcdklm} \left( \eta_{cd}^{ab} \Gamma_{am}^{kl} - \Gamma_{cd}^{ab} \eta_{am}^{kl} \right) \lambda_{cdm}^{bkl},
$$

---
## 2. One-body Flow: $df/ds$

The 1-body flow equation comes from $[\eta, H]_{1B}$.




In [8]:
# df/ds comes from [1,1]_1, [1,2]_1, [2,1]_1, [2,2]_1

print("="*60)
print("Computing one-body flow...")
print("="*60)

t0 = time.time()
comm_111 = qcombo.easyCombo(1, 1, 1, parallel=False, show_process=False, savefile=False)
comm_121 = qcombo.easyCombo(1, 2, 1, parallel=False, show_process=False, savefile=False)
comm_211 = qcombo.easyCombo(2, 1, 1, parallel=False, show_process=False, savefile=False)
comm_221 = qcombo.easyCombo(2, 2, 1, parallel=False, show_process=False, savefile=False)
t1 = time.time()

print(f"Computation time: {t1-t0:.2f}s")

Computing one-body flow...
Computation time: 1.79s


In [9]:
# Aggregate results from expr_dict and display raw expressions
comm_111_expr = 0
comm_121_expr = 0
comm_211_expr = 0
comm_221_expr = 0

for key, value in comm_111.expr_dict.items():
    comm_111_expr += value

for key, value in comm_121.expr_dict.items():
    comm_121_expr += value

for key, value in comm_211.expr_dict.items():
    comm_211_expr += value

for key, value in comm_221.expr_dict.items():
    comm_221_expr += value

# print('commutator [1,1]-1 expression:')
# jupyterDisplay(comm_111_expr)
# print('commutator [1,2]-1 expression:')
# jupyterDisplay(comm_121_expr)
# print('commutator [2,1]-1 expression:')
# jupyterDisplay(comm_211_expr)
# print('commutator [2,2]-1 expression:')
# jupyterDisplay(comm_221_expr)


In [10]:
# Restore index antisymmetry of the expressions
comm_111_expr = re_antisymmetry(comm_111_expr)
comm_121_expr = re_antisymmetry(comm_121_expr)
comm_211_expr = re_antisymmetry(comm_211_expr)
comm_221_expr = re_antisymmetry(comm_221_expr)

# print('commutator [1,1]-1 expression (antisymmetrized):')
# jupyterDisplay(comm_111_expr)
# print('commutator [1,2]-1 expression (antisymmetrized):')
# jupyterDisplay(comm_121_expr)
# print('commutator [2,1]-1 expression (antisymmetrized):')
# jupyterDisplay(comm_211_expr)
# print('commutator [2,2]-1 expression (antisymmetrized):')
# jupyterDisplay(comm_221_expr)


In [11]:
# Replace G → η (generator), H → f/Γ to obtain the flow equation
# with proper coefficients
A_tensor = find_A_tensor(comm_111_expr)
# jupyterDisplay(A_tensor)


df_ds_111_expr = replace_H_Base(replace_G_Base(comm_111_expr, r'\eta'), "f")/A_tensor
df_ds_121_expr = replace_H_Base(replace_G_Base(comm_121_expr, r'\eta'), r'\Gamma')/A_tensor/4
df_ds_211_expr = replace_H_Base(replace_G_Base(comm_211_expr, r'\eta'), "f")/A_tensor/4
df_ds_221_expr = replace_H_Base(replace_G_Base(comm_221_expr, r'\eta'), r'\Gamma')/A_tensor/16

# jupyterDisplay(df_ds_111_expr, "[η_1B, f] → 1B:")
# jupyterDisplay(df_ds_121_expr, "[η_1B, Γ] → 1B:")
# jupyterDisplay(df_ds_211_expr, "[η_2B, f] → 1B:")
# jupyterDisplay(df_ds_221_expr, "[η_2B, Γ] → 1B:")


In [12]:
print("df^a_b/ds flow equation contains:")
display(Latex(r'$$d f^{a}_{b} =  $$'))
# [1,1]-1
print('commutator [1,1]-1, lambda_1B:')
jupyterDisplay(df_ds_111_expr)
# [1,2]-1
print('commutator [1,2]-1, lambda_2B:')
jupyterDisplay(df_ds_121_expr)
# [2,1]-1
print('commutator [2,1]-1, lambda_2B:')
jupyterDisplay(df_ds_211_expr)
# [2,2]-1 — classify by λ body number
print('commutator [2,2]-1, lambda_1B:')
jupyterDisplay(filterLambdaBody(df_ds_221_expr, 1, False, False))
print('commutator [2,2]-1, lambda_2B:')
jupyterDisplay(filterLambdaBody(df_ds_221_expr, 2, False, False))



df^a_b/ds flow equation contains:


<IPython.core.display.Latex object>

commutator [1,1]-1, lambda_1B:


<IPython.core.display.Latex object>

commutator [1,2]-1, lambda_2B:


...

<IPython.core.display.Latex object>

commutator [2,2]-1, lambda_1B:


<IPython.core.display.Latex object>

commutator [2,2]-1, lambda_2B:


<IPython.core.display.Latex object>

## $df / ds$ in Reference

$$
\frac{d}{ds} f_j^i = \sum_a \left( \eta_a^i f_j^a - f_a^i \eta_j^a \right) + \sum_{ab} \left( \eta_b^a \Gamma_{aj}^{bi} - f_b^a \eta_{aj}^{bi} \right) \left( n_a - n_b \right)
$$

$$
+ \frac{1}{2} \sum_{abc} \left( \eta_{bc}^{ia} \Gamma_{ja}^{bc} - \Gamma_{bc}^{ia} \eta_{ja}^{bc} \right) \left( n_a \bar{n}_b \bar{n}_c + \bar{n}_a n_b n_c \right)
$$

$$
+ \frac{1}{4} \sum_{abcde} \left( \eta_{bc}^{ia} \Gamma_{ja}^{de} - \Gamma_{bc}^{ia} \eta_{ja}^{de} \right) \lambda_{bc}^{de} + \sum_{abcde} \left( \eta_{bc}^{ia} \Gamma_{jd}^{be} - \Gamma_{bc}^{ia} \eta_{jd}^{be} \right) \lambda_{cd}^{ae} 
$$

$$- \frac{1}{2} \sum_{abcde} \left( \eta_{jb}^{ia} \Gamma_{ae}^{cd} - \Gamma_{jb}^{ia} \eta_{ae}^{cd} \right) \lambda_{be}^{cd} + \frac{1}{2} \sum_{abcde} \left( \eta_{jb}^{ia} \Gamma_{de}^{bc} - \Gamma_{jb}^{ia} \eta_{de}^{bc} \right) \lambda_{de}^{ac},

---
## 3. Two-body Flow: $d\Gamma/ds$

The 2-body flow equation comes from $[\eta, H]_{2B}$.



In [13]:
# dΓ/ds comes from [1,2]_2, [2,1]_2, [2,2]_2
# ([1,1]_2 vanishes — no valid contraction for 1B × 1B → 2B)

print("="*60)
print("Computing two-body flow...")
print("="*60)

t0 = time.time()
comm_122 = qcombo.easyCombo(1, 2, 2, parallel=False, show_process=False, savefile=False)
comm_212 = qcombo.easyCombo(2, 1, 2, parallel=False, show_process=False, savefile=False)
comm_222 = qcombo.easyCombo(2, 2, 2, parallel=False, show_process=False, savefile=False)
t1 = time.time()

print(f"Computation time: {t1-t0:.2f}s")

Computing two-body flow...
Computation time: 1.01s


In [14]:
# Aggregate results from expr_dict and display raw expressions
comm_122_expr = 0
comm_212_expr = 0
comm_222_expr = 0

for key, value in comm_122.expr_dict.items():
    comm_122_expr += value

for key, value in comm_212.expr_dict.items():
    comm_212_expr += value

for key, value in comm_222.expr_dict.items():
    comm_222_expr += value

# print('commutator [1,2]-2 expression:')
# jupyterDisplay(comm_122_expr)
# print('commutator [2,1]-2 expression:')
# jupyterDisplay(comm_212_expr)
# print('commutator [2,2]-2 expression:')
# jupyterDisplay(comm_222_expr)


In [15]:
# Restore index antisymmetry of the expressions
comm_122_expr = re_antisymmetry(comm_122_expr)
comm_212_expr = re_antisymmetry(comm_212_expr)
comm_222_expr = re_antisymmetry(comm_222_expr)

# print('commutator [1,2]-2 expression (antisymmetrized):')
# jupyterDisplay(comm_122_expr)
# print('commutator [2,1]-2 expression (antisymmetrized):')
# jupyterDisplay(comm_212_expr)
# print('commutator [2,2]-2 expression (antisymmetrized):')
# jupyterDisplay(comm_222_expr)


In [16]:
# Replace G → η (generator), H → f/Γ to obtain the flow equation
# with proper coefficients
A_tensor = find_A_tensor(comm_122_expr)
# jupyterDisplay(A_tensor)

dG_ds_122_expr = replace_H_Base(replace_G_Base(comm_122_expr, r'\eta'), r'\Gamma')/A_tensor
dG_ds_212_expr = replace_H_Base(replace_G_Base(comm_212_expr, r'\eta'), "f")/A_tensor
dG_ds_222_expr = replace_H_Base(replace_G_Base(comm_222_expr, r'\eta'), r'\Gamma')/A_tensor/4

# jupyterDisplay(dG_ds_122_expr, "[η_1B, Γ] → 2B:")
# jupyterDisplay(dG_ds_212_expr, "[η_2B, f] → 2B:")
# jupyterDisplay(dG_ds_222_expr, "[η_2B, Γ] → 2B:")


In [17]:
print("dΓ^{ab}_{cd}/ds flow equation contains:")
display(Latex(r'$$d \Gamma^{ab}_{cd} =  $$'))
# [1,2]-2
print('commutator [1,2]-2, lambda_2B:')
jupyterDisplay(dG_ds_122_expr)
# [2,1]-2
print('commutator [2,1]-2, lambda_2B:')
jupyterDisplay(dG_ds_212_expr)
# [2,2]-2 — classify by λ body number
print('commutator [2,2]-2, lambda_1B:')
jupyterDisplay(filterLambdaBody(dG_ds_222_expr, 1, False, False))



dΓ^{ab}_{cd}/ds flow equation contains:


<IPython.core.display.Latex object>

commutator [1,2]-2, lambda_2B:


<IPython.core.display.Latex object>

commutator [2,1]-2, lambda_2B:


<IPython.core.display.Latex object>

commutator [2,2]-2, lambda_1B:


<IPython.core.display.Latex object>

## $d \Gamma /ds$ in Reference


$$\frac{d}{ds} \Gamma_{kl}^{ij} = \sum_a \left( \eta_a^i \Gamma_{kl}^{aj} + \eta_j^a \Gamma_{kl}^{ia} - \eta_k^a \Gamma_{al}^{ij} - \eta_l^a \Gamma_{ka}^{ij} - f_a^i \eta_{kl}^{aj} - f_a^j \eta_{kl}^{ia} + f_k^a \eta_{al}^{ij} + f_l^a \eta_{ka}^{ij} \right) $$
$$ + \frac{1}{2} \sum_{ab} \left( \eta_{ab}^{ij} \Gamma_{kl}^{ab} - \Gamma_{ab}^{ij} \eta_{kl}^{ab} \right) \left( 1 - n_a - n_b \right) $$
$$+ \sum_{ab} (n_a - n_b) \left( \left( \eta_{kb}^{ia} \Gamma_{la}^{jb} - \Gamma_{kb}^{ia} \eta_{la}^{jb} \right) - \left( \eta_{kb}^{ja} \Gamma_{la}^{ib} - \Gamma_{kb}^{ja} \eta_{la}^{ib} \right) \right). $$


---
## Appendix: Substituting the Brillouin Generator

For a self-contained derivation, one can substitute the explicit Brillouin generator expressions
into the flow equations. The generator expressions (from `IMSRG_Brillouin_Generator.ipynb`) are:

$$\eta^k_l = f^k_l (n_l - n_k) - \frac{1}{2}\sum_{abc} \left(\Gamma^{la}_{bc} \lambda^{ka}_{bc} - \Gamma^{ab}_{kc} \lambda^{ab}_{lc}\right)$$

$$\eta^{kl}_{mn} = \Gamma^{kl}_{mn}(\bar{n}_k\bar{n}_l n_m n_n - n_k n_l \bar{n}_m \bar{n}_n) + \cdots$$

Substituting these into the flow equations above yields the complete IMSRG(2) working equations.
